这是用于模型检验的推断部分使用Visium HD数据的处理部分：这部分将使用bin2cell完成标准数据读取，预处理和tile分割，并转化成dataset用于推断，
并将推断后的结果写入到新的csr中，产生处理后的稀疏矩阵和细胞标签。

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np
import os
import bin2cell as b2c

#create directory for stardist input/output files
os.makedirs("stardist", exist_ok=True)

In [ ]:
path = "/root/autodl-tmp/binned_outputs/square_002um"
source_image_path = "/root/autodl-tmp/Visium_HD_FF_Human_Breast_Cancer_tissue_image.tif"
spaceranger_image_path = "/root/autodl-tmp/binned_outputs/square_002um/spatial"

In [ ]:
adata = b2c.read_visium(path, 
                    source_image_path = source_image_path, 
                    spaceranger_image_path = spaceranger_image_path
                    )
adata.var_names_make_unique()
adata

In [ ]:
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_counts=1)
adata

In [ ]:
#likely to be closer to 0.3 for your data
mpp = 0.5

b2c.scaled_he_image(adata, mpp=mpp, save_path="stardist/he.tiff")

In [ ]:
# 选择ROI进行counts的分布展示
mask = ((adata.obs['array_row'] >= 1450) & 
        (adata.obs['array_row'] <= 1550) & 
        (adata.obs['array_col'] >= 250) & 
        (adata.obs['array_col'] <= 450)
       )

bdata = adata[mask]
sc.pl.spatial(bdata, color=[None, "n_counts", "n_counts_adjusted"], color_map="OrRd",
              img_key="0.5_mpp_150_buffer", basis="spatial_cropped_150_buffer")

In [ ]:
b2c.stardist(image_path="stardist/he.tiff", 
             labels_npz_path="stardist/he.npz", 
             stardist_model="2D_versatile_he", 
             prob_thresh=0.01
            )

In [ ]:
b2c.insert_labels(adata, 
                  labels_npz_path="stardist/he.npz", 
                  basis="spatial", 
                  spatial_key="spatial_cropped_150_buffer",
                  mpp=mpp, 
                  labels_key="labels_he"
                 )

In [ ]:
bdata = adata[mask]

#the labels obs are integers, 0 means unassigned
bdata = bdata[bdata.obs['labels_he']>0]
bdata.obs['labels_he'] = bdata.obs['labels_he'].astype(str)

sc.pl.spatial(bdata, color=[None, "labels_he"], img_key="0.5_mpp_150_buffer", basis="spatial_cropped_150_buffer")

In [ ]:
#the label viewer wants a crop of the processed image
#get the corresponding coordinates spanning the subset object
crop = b2c.get_crop(bdata, basis="spatial", spatial_key="spatial_cropped_150_buffer", mpp=mpp)

rendered = b2c.view_labels(image_path="stardist/he.tiff", 
                           labels_npz_path="stardist/he.npz", 
                           crop=crop
                          )
plt.imshow(rendered)

In [ ]:
# 降采样表达谱灰度图并进行整个样本的可视化展示
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt

# 2. 计算每个 Bin 的总 UMI 表达量
total_counts = np.array(adata.X.sum(axis=1)).flatten()

# 3. 获取坐标并构建二维矩阵
rows = adata.obs['array_row'].values
cols = adata.obs['array_col'].values

img_matrix = np.zeros((rows.max() + 1, cols.max() + 1), dtype=np.float32)
img_matrix[rows, cols] = total_counts

# 为了可视化对比度，进行 log1p 变换压制极值
img_log = np.log1p(img_matrix)

# 4. 绘制带有坐标轴的图像
plt.figure(figsize=(12, 12), dpi=150)
plt.imshow(img_log, cmap='gray')
plt.colorbar(label='Log1p(Total UMI)')
plt.xlabel('array_col (X)')
plt.ylabel('array_row (Y)')
plt.title('Visium HD Spatial Map')
plt.show()